In [ ]:
#!pip instal openie
#!pip install pypandoc

In [1]:
# --- Standard Library ---
import os
import time
import warnings
from datetime import datetime, timedelta
from typing import Any, Dict, List, Literal, Tuple, TypedDict

# --- Third-Party Libraries ---
import folium
import geopandas as gpd
import googlemaps
import gradio as gr
import numpy as np
import openrouteservice
import pandas as pd
import shapely.geometry
from dotenv import load_dotenv

from folium.plugins import BeautifyIcon
from geopy.geocoders import Nominatim
from geographiclib.geodesic import Geodesic
from markdown_pdf import MarkdownPdf, Section
from openai import OpenAI

from tavily import TavilyClient
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
import re
import base64

import tempfile
import shutil
import pypandoc
from datetime import datetime



# Suppress pandas warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


# --- Environment Setup ---
load_dotenv(override = True)
ORS_API_KEY = os.getenv("ORS_API_KEY")
GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
# Added for the new tab
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


ors_client = openrouteservice.Client(key=ORS_API_KEY)
gmaps_client = googlemaps.Client(key=GOOGLE_MAPS_API_KEY)
geolocator = Nominatim(user_agent="my-trip-app")

# --- LLM and Tool Initialization (for new tab) ---
openai_client = OpenAI(api_key=OPENAI_API_KEY)
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)



class Pdf_Output(TypedDict):
    city: str
    country: str
    city_guide: str
    tourist_office : str
    hotel_name: str
    hotel_description:str
    hotel_info: str
    trip_out: str
    trip_home: str
    restaurants: str
    planner: str

# Global state to hold the WbsState between Gradio calls
to_pdf: Pdf_Output = {
    'city': "",
    'country': "",
    'city_guide': "",
    'tourist_office' : "",
    'hotel_name': "",
    'hotel_description': "",
    'hotel_info' : "",
    'trip_out': "",
    'trip_home': "",
    'restaurants': "",
    'planner': "",}




# --- Constants ---
CHARGE_UP_TO_PERCENT = 90.0
AVERAGE_SPEED_KMPH = 80.0
CHARGING_TIME_MINUTES = 45.0
SEARCH_RADIUS_METERS = 2000
MAX_RESULTS_TO_PROCESS = 5
BATTERY_CAPACITY_KWH = 78.0
CONSUMPTION_KWH_PER_100KM = 17.31

# --- Helper functions ---
def remaining_battery(start_percent, distance_km):
    energy_used_kwh = (distance_km / 100) * CONSUMPTION_KWH_PER_100KM
    percent_used = (energy_used_kwh / BATTERY_CAPACITY_KWH) * 100
    return max(start_percent - percent_used, 0)

def format_time_hm(hours):
    if hours < 0: return "0 minutes"
    h, m = divmod(int(hours * 60), 60)
    if h > 0 and m > 0: return f"{h} hours and {m} minutes"
    elif h > 0: return f"{h} hours"
    else: return f"{m} minutes"

def geodesic_distance(lat1, lon1, lat2, lon2):
    """Calculate geodesic distance between two points."""
    return Geodesic.WGS84.Inverse(lat1, lon1, lat2, lon2)['s12']


# ---- Geocoding functions ---
def get_google_coords(address):
    try:
        geocode_result = gmaps_client.geocode(address)
        if geocode_result:
            location = geocode_result[0]['geometry']['location']
            return (location['lat'], location['lng'])
        return None
    except Exception as e:
        print(f"Error geocoding with Google: {e}")
        return None

def get_coordinates_from_address(address):
    try:
        location = geolocator.geocode(address)
        return (location.latitude, location.longitude) if location else get_google_coords(address)
    except Exception: return None

def get_city_from_coords(gmaps_client, lat, lon):
    try:
        reverse_geocode_result = gmaps_client.reverse_geocode((lat, lon))
        if reverse_geocode_result:
            for component in reverse_geocode_result[0]['address_components']:
                if 'locality' in component['types']: return component['long_name']
        return None
    except Exception as e:
        print(f"Error reverse geocoding: {e}")
        return None


def get_route_ors(start_coords, end_coords):
    if not start_coords or not end_coords: return None, None, None, None
    try:
        route = ors_client.directions(coordinates=[[start_coords[1], start_coords[0]], [end_coords[1], end_coords[0]]], profile="driving-car", format="geojson")
        geometry = route["features"][0]["geometry"]["coordinates"]
        distance = route["features"][0]["properties"]["segments"][0]["distance"] / 1000
        duration = route["features"][0]["properties"]["segments"][0]["duration"] / 60
        return route, geometry, distance, duration
    except Exception as e:
        print(f"Error getting route from ORS: {e}")
        return None, None, None, None

def find_stations_along_route(route_coords, stations_gdf, buffer_distance_meters=2000):
    if not route_coords or stations_gdf.empty: return []
    route_line = shapely.geometry.LineString(route_coords)
    buffered_route = route_line.buffer(buffer_distance_meters / 111000)
    nearby_stations = stations_gdf[stations_gdf.geometry.intersects(buffered_route)]
    results = []
    for _, station in nearby_stations.iterrows():
        address = f"{station.get('Address', '')}, {station.get('Zip_Code', '')} {station.get('City', '')}".strip(", ")
        results.append({"brand": station.get("Backend_Operator", "N/A"), "latitude": station.geometry.y, "longitude": station.geometry.x, "location": address})
    return results


def find_places_google(gmaps_client, location_coords, query, place_type):
    try:
        places_result = gmaps_client.places(query=query, location=location_coords, radius=SEARCH_RADIUS_METERS, type=place_type)
        places = [{'name': p.get('name'), 'address': p.get('formatted_address', 'N/A'), 'latitude': p['geometry']['location']['lat'], 'longitude': p['geometry']['location']['lng'], 'place_id': p.get('place_id'), 'rating': p.get('rating'), 'user_ratings_total': p.get('user_ratings_total')} for p in places_result.get('results', [])]
        return places
    except Exception as e:
        print(f"Error finding places with Google: {e}")
        return []

def get_google_place_details(place_id):
    if not place_id: return {}
    try:
        fields = ['website', 'formatted_phone_number', 'permanently_closed']
        place_details = gmaps_client.place(place_id=place_id, fields=fields)
        result = place_details.get('result', {})
        return {'website': result.get('website'), 'phone_number': result.get('formatted_phone_number'), 'permanently_closed': result.get('permanently_closed', False)}
    except Exception as e:
        print(f"Error fetching place details for {place_id}: {e}")
        return {}

def get_walking_distances_google(origin_coords, destinations):
    if not destinations: return destinations
    try:
        matrix_result = gmaps_client.distance_matrix(origins=[origin_coords], destinations=[(d['latitude'], d['longitude']) for d in destinations], mode="walking", units="metric")
        if matrix_result['rows']:
            for i, element in enumerate(matrix_result['rows'][0]['elements']):
                if element['status'] == 'OK':
                    destinations[i].update({'distance': element['distance']['text'], 'duration': element['duration']['text']})
                else:
                    destinations[i].update({'distance': 'N/A', 'duration': 'N/A'})
        return destinations
    except Exception as e:
        print(f"Error calculating distances with Google: {e}")
        return destinations

def generate_google_maps_url(place_id, place_type="hotel"):
    if place_id:
        return f"https://www.google.com/maps/search/?api=1&query={place_type}&query_place_id={place_id}"
    return None

# --- GenAI functions ---
# ----HELPERS-----

def clean_text(text: str) -> str:
    """
    Clean Tavily snippets for optimal LLM ingestion.
    """

    if not text:
        return ""

    # Remove URLs
    text = re.sub(r"http\S+", "", text)

    # Remove markdown links
    text = re.sub(r"\[([^\]]+)\]\((.*?)\)", r"\1", text)

    # Remove markdown formatting
    text = re.sub(r"[*_#>`~]+", " ", text)

    # Remove weird bullets/unicode artifacts
    text = re.sub(r"[•●▪■◆▶►◦∗]+", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

def tavily_search_section(section_name: str, query: str) -> tuple:
    """
    Performs optimized Tavily semantic search.
    Returns condensed high-quality context for LLM usage.
    """

    try:

        response = tavily_client.search(
            query=query,
            search_depth="advanced",
            topic="general",
            max_results=5,
            include_answer=True,
            include_raw_content=False,
            include_images=False,
        )

        chunks = []

        # Tavily synthesized answer
        if response.get("answer"):
            chunks.append(
                f"SUMMARY:\n{clean_text(response['answer'])}"
            )

        # Individual results
        for result in response.get("results", []):

            title = result.get("title", "")
            content = clean_text(result.get("content", ""))

            if not content:
                continue

            chunks.append(
                f"TITLE: {title}\n"
                f"CONTENT:\n{content}"
            )

        context = "\n\n".join(chunks)

        return section_name, context

    except Exception as e:
        return section_name, f"Search failed: {str(e)}"


# -----------------------------------------------------------------------------
# MAIN FUNCTION
# -----------------------------------------------------------------------------

def get_city_guide(city_name: str, country_name: str) -> tuple:

    try:

        gr.Info(
            f"🔍 Researching {city_name}, {country_name}...",
            duration=20
        )

        # ---------------------------------------------------------------------
        # SEMANTIC SEARCH QUERIES
        # ---------------------------------------------------------------------

        search_queries = {
            "tourism":
                f"{city_name} {country_name} top attractions relaxed weekend hidden gems neighborhoods",

            "history":
                f"{city_name} {country_name} history major historical events heritage",

            "economy":
                f"{city_name} {country_name} economy industries business profile",

            "culture":
                f"{city_name} {country_name} culture arts museums events lifestyle",

            "shopping":
                f"{city_name} {country_name} shopping centers, shopping streets, markets, boutiques, and local products",

            "sports":
                f"{city_name} {country_name} famous sports teams athletes sporting events",

            "drinks":
                f"{city_name} {country_name} local wines beers breweries wineries regional drinks",

            "restaurants":
                f"best restaurants in {city_name} {country_name} authentic {country_name}, Italian, or Croatian cuisine NO non_european cuisines.",
        }

        # ---------------------------------------------------------------------
        # PARALLEL TAVILY SEARCH
        # ---------------------------------------------------------------------

        raw_context = {}

        with ThreadPoolExecutor(max_workers=6) as executor:

            futures = {
                executor.submit(
                    tavily_search_section,
                    key,
                    query
                ): key
                for key, query in search_queries.items()
            }

            for future in as_completed(futures):

                key, context = future.result()
                raw_context[key] = context

        # ---------------------------------------------------------------------
        # STRUCTURED LLM CONTEXT
        # ---------------------------------------------------------------------

        structured_context = "\n\n".join([
            f"## {section.upper()}\n{content}"
            for section, content in raw_context.items()
        ])

        print(structured_context)

        # ---------------------------------------------------------------------
        # LLM PROMPT
        # ---------------------------------------------------------------------

        guide_prompt = f"""
            You are an expert European travel writer specialized in relaxed luxury weekend trips.
            
            Create a detailed but concise weekend guide for:
            
            CITY: {city_name}
            COUNTRY: {country_name}
            
            Use ONLY the verified search context below.
            
            SEARCH CONTEXT:
            {structured_context}
            
            IMPORTANT RULES:
            
            - Do NOT mention missing information.
            - Do NOT hallucinate fake restaurants or attractions.
            - Do NOT mention restaurants with NON european cuisines, like asin , turkish, syrain, lebanees, thai, chinese,...
            - Keep tone elegant, informative, relaxed, and practical.
            - Focus on slow travel and enjoyable city experiences.
            - Avoid adventure sports or physically intensive activities.
            - Mention famous sports teams ONLY if internationally famous.
            - Restaurant recommendations must feel realistic and current.
            - Mention regional wines and beers when relevant.
            - Use clean markdown formatting.
            
            OUTPUT FORMAT:
            
            # Discover {city_name}: Your Weekend Guide
            
            ---
            [Introduction paragraph]
            
            ---
            
            ## What is {city_name} famous for?
            
            - item
            - item
            - item
            - item
            
            ## Top Attractions & Things to Do
            
            - attraction
            - attraction
            - attraction
            - attraction
            - hiden gems and fanous neighborhoods
            
            ## Shopping & Local Finds
            
            [shopping centers, shopping areas, shopping streets, boutiques, local markets, local products]
            
            ## Culture & Leisure
            
            [Concise section]
            
            ## Wining & Dining
            
            [Local food and drink culture: describe regional cuisine, local wines, beers, spirits, and culinary traditions. No specific restaurant names or listings. Focus on what to try, local specialties, market culture, street food character, and dining atmosphere.]
            
            
            [A brief historic overview]
            
            ---
            """

        # ---------------------------------------------------------------------
        # LLM GENERATION
        # ---------------------------------------------------------------------

        gr.Info(
            "📝 Writing travel guide...",
            duration=15
        )

        response = openai_client.chat.completions.create(
            model="gpt-5.5",
            #temperature=0.4,
            max_completion_tokens=3500,
            messages=[
                {
                    "role": "system",
                    "content":
                        "You are a high-end travel editor producing accurate, readable, structured travel guides."
                },
                {
                    "role": "user",
                    "content": guide_prompt
                }
            ]
        )

        guide = response.choices[0].message.content

        gr.Info(
            "✅ Travel guide completed",
            duration=5
        )

        # return guide, structured_context
        return guide, guide

    except Exception as e:

        error_msg = f"""
            # Error generating city guide
            
            City: {city_name}
            Country: {country_name}
            
            Error:
            {str(e)}
            """

        return error_msg, ""

def tavily_context_search(
        query: str,
        max_results: int = 5,
        max_chars: int = 15000,
    ) -> str:
    """
    Search -> Extract -> Build clean context.

    Much higher quality than using Tavily snippets alone.
    """

    try:

        # --------------------------------------------------
        # Search
        # --------------------------------------------------

        search_response = tavily_client.search(
            query=query,
            search_depth="advanced",
            topic="general",
            max_results=max_results,
            include_answer=True,
            include_raw_content=False,
            include_images=False,
        )

        urls = [
            r["url"]
            for r in search_response.get("results", [])
            if r.get("url")
        ]

        chunks = []

        # Tavily synthesized answer
        if search_response.get("answer"):
            chunks.append(
                f"SUMMARY\n{clean_text(search_response['answer'])}"
            )

        if not urls:
            return "\n\n".join(chunks)

        # --------------------------------------------------
        # Extract full page content
        # --------------------------------------------------

        extract_response = tavily_client.extract(
            urls=urls,
            include_images=False,
        )

        results = extract_response.get("results", [])

        for result in results:

            title = result.get("title", "")
            raw_content = (
                result.get("raw_content")
                or result.get("content")
                or ""
            )

            cleaned = clean_text(raw_content)

            if not cleaned:
                continue

            chunks.append(
                f"TITLE: {title}\n"
                f"{cleaned[:4000]}"
            )

        context = "\n\n".join(chunks)

        return context[:max_chars]

    except Exception as e:

        print(f"Tavily extract error: {e}")
        return ""


def get_info(
        prompt: str,
        use_search: bool = True,
        model: str = "gpt-5.5",
    ):
    """
    Search -> Extract -> LLM Answer
    """

    context = ""

    if use_search:

        try:
    
            gr.Info(
                f"🔍 Researching: {prompt[:60]}...",
                duration=5
            )
    
            if any(
                keyword in prompt.lower()
                for keyword in [
                    "hotel",
                    "restaurant",
                    "charging station",
                    "tourist office"
                ]
            ):
                max_results = 1
            else:
                max_results = 5
    
            context = tavily_context_search(
                query=prompt,
                max_results=max_results,
            )
    
        except Exception as e:
    
            print(f"Tavily search error: {e}")

    system_msg = """
    You are a highly accurate AI research assistant.
    
    Use the provided web research context to answer accurately.
    
    Rules:
    - Prefer extracted source content over prior knowledge.
    - Synthesize information from multiple sources.
    - Ignore navigation menus, cookie banners and SEO spam.
    - Be concise but informative.
    - If the context is incomplete, use general knowledge cautiously.
    """

    if context:

        user_msg = f"""
        QUESTION:
        {prompt}
        
        WEB RESEARCH CONTEXT:
        {context}
        
        Provide a high-quality synthesized answer.
        """

    else:

        user_msg = prompt

    try:
        print(user_msg)
        response = openai_client.chat.completions.create(
            model=model,
            #temperature=0.4,
            #max_completion_tokens=2000,
            messages=[
                {
                    "role": "system",
                    "content": system_msg,
                },
                {
                    "role": "user",
                    "content": user_msg,
                },
            ],
        )

        print(response.choices[0].message.content)

        return response.choices[0].message.content

    except Exception as e:

        print(f"OpenAI API error: {e}")

        return f"Error generating response.\n\nDetails:\n{e}"


def find_high_end_hotels_in_city_center(city_name, radius=1500):
    """
    Finds 4 and 5-star hotels near a city's center, filters for those within
    1500 meters, and includes the hotel's website.
    """
    try:
        print(f"Geocoding the center of '{city_name}'...")
        geocode_result = gmaps_client.geocode(f"city center of {city_name}")
        if not geocode_result:
            print(f"Could not find the city center for '{city_name}'.")
            return []
        
        location_coords = geocode_result[0]['geometry']['location']
        lat_center = location_coords['lat']
        lon_center = location_coords['lng']
    except Exception as e:
        print(f"An error occurred during geocoding: {e}")
        return []

    all_hotels = []
    seen_place_ids = set()
    for star_rating in [5, 4]:
        query = f"{star_rating} star hotels in {city_name}"
        print(f"Searching for '{query}' near the city center...")
        try:
            places_result = gmaps_client.places(
                query=query, location=location_coords, radius=radius, type='lodging'
            )
            for place in places_result.get('results', []):
                place_id = place.get('place_id')
                if place_id and place_id not in seen_place_ids:
                    hotel_location = place.get('geometry', {}).get('location', {})
                    lat_hotel = hotel_location.get('lat')
                    lon_hotel = hotel_location.get('lng')

                    if lat_hotel is not None and lon_hotel is not None:
                        # Calculate the precise distance from the city center
                        distance = geodesic_distance(lat_center, lon_center, lat_hotel, lon_hotel)

                        # Only include hotels within 3000 meters
                        if distance <= 3000:
                            try:
                                details = gmaps_client.place(place_id=place_id, fields=['website', 'formatted_address'])
                                website = details.get('result', {}).get('website')
                                address = details.get('result', {}).get('formatted_address', place.get('vicinity', 'N/A'))
                            except Exception as e:
                                website, address = None, place.get('vicinity', 'N/A')

                            all_hotels.append({
                                'name': place.get('name'),
                                'address': address,
                                'latitude': lat_hotel,
                                'longitude': lon_hotel,
                                'place_id': place_id,
                                'star_rating': f"{star_rating}-star",
                                'review_rating': place.get('rating', 'N/A'),
                                'website': website,
                                'reviews_total': place.get('user_ratings_total', 0)
                            })
                            seen_place_ids.add(place_id)
        except Exception as e:
            print(f"An error occurred while searching for '{query}': {e}")
            
    return all_hotels
        

def find_parkings_in_city_center(city_name, radius=1000):
    """Finds parkings near a city's center, including the website."""
    try:
        print(f"Geocoding the center of '{city_name}'...")
        geocode_result = gmaps_client.geocode(f"city center of {city_name}")
        if not geocode_result:
            print(f"Could not find the city center for '{city_name}'.")
            return []
        
        location_coords = geocode_result[0]['geometry']['location']
        lat_center = location_coords['lat']
        lon_center = location_coords['lng']
    except Exception as e:
        print(f"An error occurred during geocoding: {e}")
        return []

    all_parkings = []
    seen_place_ids = set()

    query = f"garage in {city_name}"
    print(f"Searching for '{query}' near the city center...")
    try:
        places_result = gmaps_client.places(
            query=query, location=location_coords, radius=radius, type='parking'
        )
        for place in places_result.get('results', []):
            place_id = place.get('place_id')
            if place_id and place_id not in seen_place_ids:
                parking_location = place.get('geometry', {}).get('location', {})
                lat_parking = parking_location.get('lat')
                lon_parking = parking_location.get('lng')

                if lat_parking is not None and lon_parking is not None:
                    # Calculate the precise distance from the city center
                    distance = geodesic_distance(lat_center, lon_center, lat_parking, lon_parking)
                    # Only include hotels within 3000 meters
                    if distance <= 3000:
                        try:
                            details = gmaps_client.place(place_id=place_id, fields=['website', 'formatted_address'])
                            website = details.get('result', {}).get('website')
                            address = details.get('result', {}).get('formatted_address', place.get('vicinity', 'N/A'))
                        except Exception as e:
                            website, address = None, place.get('vicinity', 'N/A')
        
                        all_parkings.append({
                            'name': place.get('name'), 'address': address,
                            'latitude': place.get('geometry', {}).get('location', {}).get('lat'),
                            'longitude': place.get('geometry', {}).get('location', {}).get('lng'),
                            'place_id': place_id, 
                            'review_rating': place.get('rating', 'N/A'), 'website': website
                        })
                        seen_place_ids.add(place_id)
    except Exception as e:
        print(f"An error occurred while searching for '{query}': {e}")
    return all_parkings

def generate_city_map(city_name, hotel_list=None, parking_list=None, tourist_office=None):
    """
    Generates an interactive map for a city, optionally including hotels, parkings,  and a tourist office.
    """
    hotel_list = hotel_list or []
    parking_list = parking_list or []
    
    # Center map on the city, tourist office, or average hotel location
    if tourist_office:
        center_lat, center_lng = tourist_office['latitude'], tourist_office['longitude']
    elif hotel_list:
        center_lat = sum(h['latitude'] for h in hotel_list) / len(hotel_list)
        center_lng = sum(h['longitude'] for h in hotel_list) / len(hotel_list)
    else:
        city_coords = get_google_coords(city_name)
        if not city_coords: return create_initial_map() # Fallback
        center_lat, center_lng = city_coords
        
    m = folium.Map(location=[center_lat, center_lng], zoom_start=14, tiles="OpenStreetMap")

    # Add Tourist Office Marker
    if tourist_office:
        info_popup_html=f"<b>{tourist_office['name']}</b><br>{tourist_office['address']}<br>"
        links = []
        if tourist_office.get('website'):
            links.append(f'<a href="{tourist_office["website"]}" target="_blank">Visit Website</a>')
        maps_url = generate_google_maps_url(tourist_office['place_id'])
        if maps_url:
            links.append(f'<a href="{maps_url}" target="_blank">View on Google Maps</a>')
        info_popup_html += " | ".join(links)
        folium.Marker(
            location=[tourist_office['latitude'], tourist_office['longitude']],
            tooltip=tourist_office['name'],           
            popup=folium.Popup(info_popup_html, max_width=400),
            icon=BeautifyIcon(
                icon="info", 
                prefix='fa', 
                icon_shape="circle",
                icon_size=[25,25],
                text_color='white', 
                background_color='darkgreen',
                border_color='darkgreen'),
        ).add_to(m)

        to_pdf['tourist_office'] = info_popup_html

    # Add Hotel Markers
    for hotel in hotel_list:
        hotel_color = 'darkred' if hotel['star_rating'] == '5-star' else 'orange'
        hotel_popup_html = f"<b>{hotel['name']}</b><hr style='margin: 4px;'><b>Class:</b> {hotel['star_rating']}<br>"
        hotel_popup_html += f"Rating: {hotel['review_rating']} ({hotel['reviews_total']} reviews)<br>"
        links = []
        if hotel.get('website'):
            links.append(f'<a href="{hotel["website"]}" target="_blank">Visit Website</a>')
        maps_url = generate_google_maps_url(hotel['place_id'])
        if maps_url:
            links.append(f'<a href="{maps_url}" target="_blank">View on Google Maps</a>')
        hotel_popup_html += " | ".join(links)
        
        folium.Marker(
            location=[hotel['latitude'], hotel['longitude']],
            tooltip=f"{hotel['name']} ({hotel['star_rating']})",
            popup=folium.Popup(hotel_popup_html, max_width=400),
            icon=BeautifyIcon(icon="bed", 
                              prefix='fa', 
                              icon_shape="circle",
                              icon_size=[25,25],
                              text_color='#FFFFFF', 
                              border_color=hotel_color,
                              background_color=hotel_color),
        ).add_to(m)


    for parking in parking_list:
    
        parking_popup_html = f"<b>{parking['name']}</b><br><b>Rating:</b> {parking['review_rating']}<br><hr style='margin: 4px;'>"
        links = []
        if parking.get('website'):
            links.append(f'<a href="{parking["website"]}" target="_blank">Visit Website</a>')
        
        # FIX: Add the place_type argument here
        maps_url = generate_google_maps_url(parking['place_id'], place_type="parking") 
    
        if maps_url:
            links.append(f'<a href="{maps_url}" target="_blank">View on Google Maps</a>')
        parking_popup_html += " | ".join(links)


        
        folium.Marker(
            location=[parking['latitude'], parking['longitude']],
            tooltip=f"{parking['name']} ({parking['review_rating']})",
            popup=folium.Popup(parking_popup_html, max_width=400),
            icon=BeautifyIcon(icon="square-parking", 
                              prefix='fa', 
                              icon_shape="circle",
                              text_color='#FFFFFF', 
                              border_color='#312F5E',
                              background_color='#312F5E'),
        ).add_to(m)
        
    return m._repr_html_()


# --- Gradio UI Functions ---

def create_initial_map():
    return folium.Map(location=[51.1, 4.4], zoom_start=7)._repr_html_()

# --- Functions for City Explorer Tab ---

def show_city_info(city_name, country_name):
    if not city_name:
        raise gr.Error("Please enter a city name.")
    
    # 1. Get City Guide
    guide,search_content = get_city_guide(city_name, country_name)
    
    to_pdf['city'] =  city_name
    to_pdf['country']= country_name
    to_pdf['city_guide'] = guide
    
    # 2. Find Tourist Information
    tourist_office_data = None
    try:
        places_result = gmaps_client.places(query=f"tourist information office in {city_name}")
        if places_result and places_result['results']:
            office = places_result['results'][0]
            office_id = office.get('place_id')
            details = gmaps_client.place(place_id=office_id, fields=['website', 'formatted_address'])
            website = details.get('result', {}).get('website')
            #address = details.get('result', {}).get('formatted_address', place.get('vicinity', 'N/A'))
            tourist_office_data = {
                'name': office.get('name'),
                'address': office.get('formatted_address'),
                'place_id': office_id,
                'latitude': office['geometry']['location']['lat'],
                'longitude': office['geometry']['location']['lng'],
                'website': website
            }
    except Exception as e:
        print(f"Could not find tourist office: {e}")

    # 3. Generate Map
    city_map_html = generate_city_map(city_name, tourist_office=tourist_office_data)
    
    return (
        guide,
        search_content,
        city_map_html, 
        tourist_office_data,
        gr.Markdown(visible=True),
        gr.Markdown(visible=True),
        gr.Button(visible=True), # Show hotel search button
        gr.Markdown(visible=True),
        gr.Markdown(visible=True),
        gr.Button(visible=True), # Show parking search button
        gr.Radio(visible=False, choices=[], value=None), # Hide hotel list
        gr.Markdown("") # Clear selection confirmation
    )

def search_and_display_hotels(city_name, tourist_office_data, all_parkings_data):
    if not city_name:
        raise gr.Error("City name is missing. Please search for a city first.")
        
    gr.Info(f"🔍 Searching for hotels in {city_name}...")
    hotels = find_high_end_hotels_in_city_center(city_name)
    
    if not hotels:
        gr.Warning("No high-end hotels found in the city center.")
        return gr.Radio(visible=False, choices=[], value=None), None, 
        gr.HTML(value=generate_city_map(city_name, tourist_office=tourist_office_data, parking_list= all_parkings_data))

    hotel_map_html = generate_city_map(city_name, hotel_list=hotels, tourist_office=tourist_office_data, parking_list= all_parkings_data)
    
    hotel_names = [f"{hotel['name']} ({hotel['star_rating']})" for hotel in hotels]

    gr.Info(f"✅  Finished hotels list...")
    
    return (
        gr.Radio(choices=hotel_names, value=None, label="Select a Hotel", visible=True),
        hotels, # Save the full hotel data to state
        hotel_map_html
    )

def store_and_update_hotel(selected_hotel_name, all_hotels_data):
    if not selected_hotel_name or not all_hotels_data:
        return "", gr.Textbox(value=""), gr.Textbox(value=""), gr.Textbox(value=""), None
        
    # Find the full data for the selected hotel
    selected_hotel_info = next((h for h in all_hotels_data if f"{h['name']} ({h['star_rating']})" == selected_hotel_name), None)

    if selected_hotel_info:
        gr.Info(f"🔍 Searching info for {selected_hotel_info['name']}...")
        info_hotel = get_info(f"Give a brief description of the hotel {selected_hotel_info['name']} at {selected_hotel_info['address']}")
        
        confirmation_md = f"**Selected:** You've chosen `{selected_hotel_info['name']}`\n\n"
        confirmation_md +=f"{info_hotel}"
        
        to_pdf['hotel_name'] = f"{selected_hotel_info['name']} at the address {selected_hotel_info['address']}"
        to_pdf['hotel_description']= info_hotel
        
        links=[]
        if selected_hotel_info.get('website'):
            links.append(f'<a href="{selected_hotel_info["website"]}" target="_blank">Visit Website</a>')
        maps_url = generate_google_maps_url(selected_hotel_info['place_id'])
        if maps_url:
            links.append(f'<a href="{maps_url}" target="_blank">View on Google Maps</a>')
        hotel_popup_html = " | ".join(links)

        to_pdf['hotel_info']= hotel_popup_html
        
        # This returns a Gradio Update object to change the value of another component
        return confirmation_md, gr.Textbox(value=selected_hotel_info['address']), gr.Textbox(value=selected_hotel_info['address']), gr.Textbox(value=selected_hotel_info['address']), selected_hotel_info

    return "", gr.Textbox(value=""), gr.Textbox(value=""), gr.Textbox(value=""), None

def search_and_display_parkings(city_name, tourist_office_data, all_hotels_data):
    if not city_name:
        raise gr.Error("City name is missing. Please search for a city first.")
        
    gr.Info(f"🔍 Searching for parkings in {city_name}...")
    parkings = find_parkings_in_city_center(city_name)
    
    if not parkings:
        gr.Warning("No parkingss found in the city center.")
        return None, gr.HTML(value=generate_city_map(city_name, tourist_office=tourist_office_data, hotel_list=all_hotels_data))

    parking_map_html = generate_city_map(city_name, parking_list=parkings, hotel_list=all_hotels_data, tourist_office=tourist_office_data)

    gr.Info(f"✅  Finished parkings list...")
    
    return  parkings, parking_map_html
    
# --- Functions for EV Planner Tab ---

def generate_trip_map(start_address, end_address, start_battery_str):
    gr.Info(f"🔍 Calculating route...", duration=5)
    if not all([start_address, end_address, start_battery_str]):
        raise gr.Error("Please provide a start, destination, and battery percentage.")
    try:
        start_percent = float(start_battery_str)
        if not (0 <= start_percent <= 100): raise ValueError()
    except ValueError:
        raise gr.Error("Invalid battery percentage. Must be a number between 0 and 100.")

    start_coords, end_coords = get_coordinates_from_address(start_address), get_coordinates_from_address(end_address)
    if not start_coords: raise gr.Error(f"Could not find coordinates for: {start_address}")
    if not end_coords: raise gr.Error(f"Could not find coordinates for: {end_address}")
    
    route, route_coords, distance, _ = get_route_ors(start_coords, end_coords)
    if not route: raise gr.Error("Could not calculate an initial route.")
     
    arrival_capacity = remaining_battery(start_percent, distance)

    gr.Info(f"🔍 Searching HPC charging station...", duration=5)
    
    try:
        df_all_stations = pd.read_csv('unique_locations.csv',sep=';')
        brands = ['Circle K', 'Fastned', 'Ionity', 'BP', 'Shell','EnBW Mobility','E.ON Drive','Allego']
        df_filtered = df_all_stations[df_all_stations['Backend_Operator'].isin(brands)].copy()
        gdf_stations = gpd.GeoDataFrame(df_filtered, geometry=gpd.points_from_xy(df_filtered.Longitude, df_filtered.Latitude), crs="EPSG:4326")
        found_stations = find_stations_along_route(route_coords, gdf_stations, 2000)
    except FileNotFoundError:
        found_stations = []
        gr.Warning("'unique_locations.csv' not found. Charging station search is disabled.")

    m = folium.Map(location=start_coords, zoom_start=7)
    folium.GeoJson(route, name="Route", style_function=lambda x: {'color': 'rgb(0,101,52)', 'weight': 5}).add_to(m)
    folium.Marker(location = start_coords,
              icon=BeautifyIcon(icon="home", 
                                 prefix='fa',
                                 icon_shape="circle",
                                 icon_size=[25,25],
                                 border_color='purple',
                                 text_color="#009000",
                                 background_color='yellow'),
              tooltip=f"departure").add_to(m)
    folium.Marker(location = end_coords,
              icon=BeautifyIcon(icon="flag-checkered", 
                                 prefix='fa',
                                 icon_shape="circle",
                                 icon_size=[25,25],
                                 border_color='purple',
                                 text_color="#000000",
                                 background_color='yellow'),
              tooltip=f"destination").add_to(m)
    
    route_info, battery_info = f"Initial route: {distance:.2f} km.", f"Estimated arrival without stops: **{arrival_capacity:.2f}%**."
    
    station_choices = []
    if not found_stations:
        station_info, recalc_button_visibility = "No charging stations from your brands found.", gr.Button(visible=False)
    else:
        station_info, recalc_button_visibility = f"Found {len(found_stations)} stations. Select one or more for each leg of your trip.", gr.Button(visible=True)
        for i, st in enumerate(found_stations):
            folium.Marker(location = [st['latitude'], st['longitude']],
                          icon=BeautifyIcon(icon="charging-station", 
                                             prefix='fa',
                                             icon_shape="circle",
                                             border_color='purple',
                                             text_color="#007799",
                                             background_color='yellow'),
                          tooltip=f"{st['brand']} - {st['location']}").add_to(m)
            station_choices.append(f"{i+1}. {st['brand']} - {st['location']}")
            
    status = f"{route_info} {battery_info}\n\n{station_info}"
    checkbox_update_out = gr.CheckboxGroup(choices=station_choices, visible=True, value=[], label="Select Stops for Way Out")
    checkbox_update_home = gr.CheckboxGroup(choices=station_choices, visible=True, value=[], label="Select Stops for Way Home")

    gr.Info(f"✅  Finished route planning...", duration=5)
    
    return m._repr_html_(), checkbox_update_out, checkbox_update_home, status, found_stations, recalc_button_visibility

def plan_trip_with_stops(selected_stations_out_str, selected_stations_home_str, all_found_stations, start_address, end_address, start_battery_str):
    gr.Info(f"🔍 Calculating detailed trip plan way out...", duration=25)

    if not selected_stations_out_str: raise gr.Error("Please select at least one charging station for the way out.")
    start_percent = float(start_battery_str)
    start_coords, end_coords = get_coordinates_from_address(start_address), get_coordinates_from_address(end_address)

    # --- PART 1: WAY OUT CALCULATION ---
    m_out = folium.Map(location=start_coords, zoom_start=7)
    status_out = "## Outbound Journey\n\n"
    charging_stations_out = []

    selected_stations_out = [all_found_stations[int(s.split('.')[0]) - 1] for s in selected_stations_out_str]
    ordered_stops_out = []
    current_point = start_coords
    temp_stations_list = selected_stations_out.copy()
    while temp_stations_list:
        next_stop = min(temp_stations_list, key=lambda s: get_route_ors(current_point, (s['latitude'], s['longitude']))[2] or float('inf'))
        ordered_stops_out.append(next_stop)
        temp_stations_list.remove(next_stop)
        current_point = (next_stop['latitude'], next_stop['longitude'])

    waypoints_out = [(f"{start_address}", start_coords)] + [(f"{s['brand']} - {s['location']}", (s['latitude'], s['longitude'])) for s in ordered_stops_out] + [(f"{end_address}", end_coords)]

    current_battery, total_distance_out, total_driving_hours_out = start_percent, 0, 0
    final_arrival_battery = 0
    leg_colors = ['blue', 'green', 'purple', 'orange', 'darkred']

    # Build leg table
    leg_rows_out = []
    for i in range(len(waypoints_out) - 1):
        (start_name, start_coords_leg), (end_name, end_coords_leg) = waypoints_out[i], waypoints_out[i+1]
        route, _, dist, _ = get_route_ors(start_coords_leg, end_coords_leg)
        if not route: continue

        total_distance_out += dist
        total_driving_hours_out += dist / AVERAGE_SPEED_KMPH
        arrival_battery = remaining_battery(current_battery, dist)

        short_start = start_name.split(" - ")[0] if " - " in start_name else start_name.split(",")[0]
        short_end = end_name.split(" - ")[0] if " - " in end_name else end_name.split(",")[0]
        leg_rows_out.append(f"| {i+1} | {short_start} → {short_end} | {dist:.1f} km | {arrival_battery:.1f}% |")

        time.sleep(1)
        folium.GeoJson(route, name=f"Leg {i+1}", style_function=lambda x, c=leg_colors[i % len(leg_colors)]: {'color': c}).add_to(m_out)

        icon_start = BeautifyIcon(icon="home", prefix='fa', icon_shape="circle", border_color='purple', text_color="#009000", background_color='yellow') if i == 0 else BeautifyIcon(icon="charging-station", prefix='fa', icon_shape="circle", border_color='purple', text_color="#007799", background_color='yellow')
        folium.Marker(start_coords_leg, tooltip=f"{i}. {start_name}", icon=icon_start).add_to(m_out)

        if i < len(waypoints_out) - 2:
            current_battery = CHARGE_UP_TO_PERCENT
            charging_stations_out.append(f"- **{ordered_stops_out[i]['brand']}** — {ordered_stops_out[i]['location']}")
        else:
            final_arrival_battery = arrival_battery
            folium.Marker(end_coords_leg, tooltip=f"Destination: {end_name}", icon=BeautifyIcon(icon="flag-checkered", prefix='fa', icon_shape="circle", border_color='purple', text_color="#000000", background_color='yellow')).add_to(m_out)

    # Output table
    status_out += "| Leg | From → To | Distance | Arrival Battery |\n"
    status_out += "|-----|-----------|----------|-----------------|\n"
    status_out += "\n".join(leg_rows_out)

    # Charging stations used
    if charging_stations_out:
        status_out += "\n\n**Charging stations used:**\n"
        status_out += "\n".join(charging_stations_out)

    # Trip summary
    total_charge_hours_out = len(ordered_stops_out) * (CHARGING_TIME_MINUTES / 60.0)
    total_trip_hours_out = total_driving_hours_out + total_charge_hours_out
    departure_time = datetime.now().replace(hour=17, minute=0) - timedelta(hours=total_trip_hours_out)

    status_out += f"\n\n**Trip summary:** {total_distance_out:.1f} km | {format_time_hm(total_trip_hours_out)} ({format_time_hm(total_driving_hours_out)} driving + {format_time_hm(total_charge_hours_out)} charging) | Arrival battery: {arrival_battery:.1f}% | Depart by: {departure_time.strftime('%I:%M %p')}\n"

    # --- PART 2: WAY HOME CALCULATION ---
    gr.Info(f"🔍 Calculating detailed trip plan way home...", duration=25)
    m_home = folium.Map(location=end_coords, zoom_start=7)
    status_home = "## Return Journey\n\n"
    charging_stations_home = []
    home_start_percent = final_arrival_battery

    if not selected_stations_home_str:
        status_home += "No charging stops selected for the return trip.\n"
        route_home, _, dist_home, _ = get_route_ors(end_coords, start_coords)
        if route_home:
            arrival_battery_home = remaining_battery(home_start_percent, dist_home)
            status_home += f"- **Direct route home:** {dist_home:.2f} km\n"
            status_home += f"- **Estimated arrival battery:** {arrival_battery_home:.1f}%"
            folium.GeoJson(route_home, name="Return Route", style_function=lambda x: {'color': 'red'}).add_to(m_home)
            folium.Marker(end_coords, tooltip="Home Departure", icon=BeautifyIcon(icon="home", prefix='fa', icon_shape="circle", border_color='purple', text_color="#009000", background_color='yellow')).add_to(m_home)
            folium.Marker(start_coords, tooltip="Final Arrival", icon=BeautifyIcon(icon="flag-checkered", prefix='fa', icon_shape="circle", border_color='purple', text_color="#000000", background_color='yellow')).add_to(m_home)
    else:
        selected_stations_home = [all_found_stations[int(s.split('.')[0]) - 1] for s in selected_stations_home_str]
        ordered_stops_home = []
        current_point_home = end_coords
        temp_stations_list_home = selected_stations_home.copy()
        while temp_stations_list_home:
            next_stop = min(temp_stations_list_home, key=lambda s: get_route_ors(current_point_home, (s['latitude'], s['longitude']))[2] or float('inf'))
            ordered_stops_home.append(next_stop)
            temp_stations_list_home.remove(next_stop)
            current_point_home = (next_stop['latitude'], next_stop['longitude'])

        waypoints_home = [(f"{end_address}", end_coords)] + [(f"{s['brand']} - {s['location']}", (s['latitude'], s['longitude'])) for s in ordered_stops_home] + [(f"{start_address}", start_coords)]

        current_battery_home, total_distance_home, total_driving_hours_home = home_start_percent, 0, 0

        # Build leg table for return
        leg_rows_home = []
        for i in range(len(waypoints_home) - 1):
            (start_name, start_coords_leg), (end_name, end_coords_leg) = waypoints_home[i], waypoints_home[i+1]
            route, _, dist, _ = get_route_ors(start_coords_leg, end_coords_leg)
            if not route: continue

            total_distance_home += dist
            total_driving_hours_home += dist / AVERAGE_SPEED_KMPH
            arrival_battery = remaining_battery(current_battery_home, dist)

            short_start = start_name.split(" - ")[0] if " - " in start_name else start_name.split(",")[0]
            short_end = end_name.split(" - ")[0] if " - " in end_name else end_name.split(",")[0]
            leg_rows_home.append(f"| {i+1} | {short_start} → {short_end} | {dist:.1f} km | {arrival_battery:.1f}% |")

            time.sleep(1)
            folium.GeoJson(route, name=f"Leg {i+1}", style_function=lambda x, c=leg_colors[i % len(leg_colors)]: {'color': c}).add_to(m_home)

            icon_start_home = BeautifyIcon(icon="home", prefix='fa', icon_shape="circle", border_color='purple', text_color="#009000", background_color='yellow') if i == 0 else BeautifyIcon(icon="charging-station", prefix='fa', icon_shape="circle", border_color='purple', text_color="#007799", background_color='yellow')
            folium.Marker(start_coords_leg, tooltip=f"{i}. {start_name}", icon=icon_start_home).add_to(m_home)

            if i < len(waypoints_home) - 2:
                current_battery_home = CHARGE_UP_TO_PERCENT
                charging_stations_home.append(f"- **{ordered_stops_home[i]['brand']}** — {ordered_stops_home[i]['location']}")
            else:
                folium.Marker(end_coords_leg, tooltip=f"Final Arrival: {end_name}", icon=BeautifyIcon(icon="flag-checkered", prefix='fa', icon_shape="circle", border_color='purple', text_color="#000000", background_color='yellow')).add_to(m_home)

        # Output table
        status_home += "| Leg | From → To | Distance | Arrival Battery |\n"
        status_home += "|-----|-----------|----------|-----------------|\n"
        status_home += "\n".join(leg_rows_home)

        # Charging stations used
        if charging_stations_home:
            status_home += "\n\n**Charging stations used:**\n"
            status_home += "\n".join(charging_stations_home)

        total_charge_hours_home = len(ordered_stops_home) * (CHARGING_TIME_MINUTES / 60.0)
        total_trip_hours_home = total_driving_hours_home + total_charge_hours_home
        arrival_time = datetime.now().replace(hour=15, minute=0) + timedelta(hours=total_trip_hours_home)

        status_home += f"\n\n**Trip summary:** {total_distance_home:.1f} km | {format_time_hm(total_trip_hours_home)} ({format_time_hm(total_driving_hours_home)} driving + {format_time_hm(total_charge_hours_home)} charging) | Arrival battery: {arrival_battery:.1f}% | Depart at 3:00 PM, arrive by: {arrival_time.strftime('%I:%M %p')}\n"

    gr.Info(f"✅ Finished trip planning.", duration=5)
    to_pdf['trip_out'] = status_out
    to_pdf['trip_home'] = status_home

    return m_out._repr_html_(), status_out, m_home._repr_html_(), status_home


def reset_planning_ui_full():
    return (
        gr.CheckboxGroup(visible=False, value=[]), 
        gr.CheckboxGroup(visible=False, value=[]), 
        gr.Button(visible=False), 
        gr.Markdown(""), # initial status
        gr.Markdown(""), # way out
        gr.Markdown(""), # way home
        gr.HTML(),       # way out map
        gr.HTML()        # way home map
    )

def generate_restaurant_map_with_link(hotel_address, selected_cuisines, hotel):
    """
    Generates a Folium map with markers for a hotel and nearby restaurants based on selected cuisine types.

    Args:
        hotel_address (str): The address of the hotel.
        selected_cuisines (list): A list of strings representing the desired cuisine types.
    
    Returns:
        tuple: A tuple containing the HTML representation of the map and a formatted string of results.
    """

    if not hotel_address:
        raise gr.Error("Please provide a hotel address.")
    hotel_coords = get_google_coords(hotel_address)
    if not hotel_coords:
        raise gr.Error(f"Could not find coordinates for: {hotel_address}")

    # --- Initialize Map and Hotel Marker ---
    m = folium.Map(location=hotel_coords, zoom_start=14)
    folium.Marker(hotel_coords, 
      tooltip=f"Hotel", 
      icon=BeautifyIcon(icon="home", 
                     prefix='fa',
                     icon_shape="circle",
                     icon_size=[25,25],
                     border_color='blue',
                     text_color="white",
                     background_color='blue')
     ).add_to(m)
    
    results_list=""
 
    # --- Process Restaurants for Each Selected Cuisine ---
    if selected_cuisines:
        city = get_city_from_coords(gmaps_client, hotel_coords[0], hotel_coords[1])     
        # Define a list of colors for map markers
        cuisine_colors = ['red', 'green', 'purple', 'orange', 'darkred', 'cadetblue', 'darkpurple', 'pink', 'darkblue']

        for i, cuisine in enumerate(selected_cuisines):
            
            gr.Info(f"🔍 Searching for {cuisine} restaurants...", duration=30)
            
            # --- Search for Restaurants of the current cuisine type ---
            query_parts = [part for part in [cuisine, "restaurant", (f"in {city}" if city else None)] if part]
            final_query = " ".join(query_parts)

            restaurants = find_places_google(gmaps_client, hotel_coords, query=final_query, place_type="restaurant")
            restaurants_to_process = get_walking_distances_google(hotel_coords, restaurants[:MAX_RESULTS_TO_PROCESS])
            print(restaurants_to_process)
            
            for r in restaurants_to_process:

                r.update(get_google_place_details(r['place_id']))

            # --- Filter: exclude permanently closed and low review counts ---
            filtered = []
            for r in restaurants_to_process:
                if r.get('permanently_closed'):
                    continue
                reviews = r.get('user_ratings_total', 0) or 0
                if reviews < 50:
                    continue
                if reviews < 100 and r.get('rating', 0) or 0 < 4.5:
                    continue
                filtered.append(r)
            restaurants_to_process = filtered

            # --- Group results by cuisine ---
            results_list += f"\n## {cuisine} Restaurants:\n"
            results_list += f"---\n"

            if not restaurants_to_process:
                results_list += f"- No '{cuisine}' restaurants found nearby.\n"
                continue # Skip to the next cuisine

            marker_color = cuisine_colors[i % len(cuisine_colors)] # Cycle through colors

            for r in restaurants_to_process:
                distance = geodesic_distance(hotel_coords[0], hotel_coords[1], r['latitude'], r['longitude'])

                if (r['name'] not in results_list) and (distance <= 1500):  # max 20 min walk (~1.5 km)
                    maps_url = generate_google_maps_url(r.get('place_id'), place_type='restaurant')

                    # --- Create Popup Content ---
                    popup_html = (f"<b>{r['name']} ({cuisine})</b><br>"
                                  f"Address: {r.get('address')}<br>"
                                  f"Rating: {r.get('rating', 'N/A')} ({r.get('user_ratings_total', 0)} reviews)<br>"
                                  f"Walk: {r.get('duration', 'N/A')} ({r.get('distance', 'N/A')})")

                    if r.get('website'):
                        popup_html += f"<br><a href='{r['website']}' target='_blank'>Website</a>"
                    if maps_url:
                        popup_html += f"<br><a href='{maps_url}' target='_blank'>View on Google Maps</a>"

                    # --- Add Marker to Map ---
                    folium.Marker([r['latitude'], r['longitude']],
                                  tooltip=f"{r['name']} ({cuisine})",
                                  popup=folium.Popup(popup_html, max_width=400),
                                  icon=BeautifyIcon(icon="utensils",
                                                 prefix='fa',
                                                 icon_shape="circle",
                                                 border_color=marker_color,
                                                 text_color="#FFFFFF",
                                                 background_color=marker_color)
                                 ).add_to(m)

                    # --- Add Compact Details to Text Results ---
                    results_list += f"**{r['name']}** ({cuisine})\n\n"
                    results_list += f"- *Address:* {r.get('address')}\n"
                    results_list += f"- *Walk:* {r.get('duration', 'N/A')} ({r.get('distance', 'N/A')})\n"
                    results_list += f"- *Rating:* {r.get('rating', 'N/A')} ({r.get('user_ratings_total', 0)} reviews)\n"
                    if r.get('website'):
                        results_list += f"- *Website:* [{r['name']}]({r['website']})\n"
                    if maps_url:
                        results_list += f"- *Google Maps:* [View on Map]({maps_url})\n"
                    results_list += f"---\n"
    to_pdf['restaurants'] = results_list
    return m._repr_html_(), results_list



def get_suggestions(hotel_address, selected_hotel, restaurant_list, search_content):
    if not hotel_address:
        raise gr.Error("Please provide a hotel address.")
    hotel_coords = get_google_coords(hotel_address)
    if not hotel_coords:
        raise gr.Error(f"Could not find coordinates for: {hotel_address}")
        
    city = get_city_from_coords(gmaps_client, hotel_coords[0], hotel_coords[1])

    hotel_name = selected_hotel['name']
    hotel_address = selected_hotel['address']

    # --- Initialize Map and Hotel Marker ---
    m = folium.Map(location=hotel_coords, zoom_start=14)
    folium.Marker(hotel_coords, 
                  tooltip=f"Hotel", 
                  icon=BeautifyIcon( icon="home", 
                                     prefix='fa',
                                     icon_shape="circle",
                                     icon_size=[25,25],
                                     border_color='blue',
                                     text_color="white",
                                     background_color='blue')).add_to(m)

    # --- PROMPT TEMPLATES ---
    # These templates are the building blocks for our final, dynamic prompt.
    
    EVENING_TEMPLATE = """
    - First, a pre-dinner drink at the bar of the {hotel} at {hotel_address} for about 1 hour.
    - From the bar, outline a short, relaxed walk (15-25 minutes) designed to give an impression of the city at night. This walk could pass by a key illuminated landmark, a lively square, or a particularly charming street.
    - The walk should lead to a restaurant for dinner. Please provide 2-3 restaurant suggestions from {restaurant_list}. Dinner should last about 3 hours.
    - The entire evening's itinerary, including the drink, walk, and dinner, should be timed for a comfortable return to the hotel between 10:00 PM and 11:00 PM.
    - Upon returning to the hotel, there is time for a final chat and a drink before sleeping.
    """
    
    ALL_DAY_TEMPLATE = """
    - Give me a suggestion of places I can visit in the city of {city}.
    - It should be a mix of landmarks, shopping places, and neighborhoods
    - The main tourist information office should always be included as a stop.
    - If possible, include local markets or flea markets.
    - This should be a relaxed walking tour possible on a Saturday from 11 AM until 5 PM, starting from and returning to the hotel: {hotel} at {hotel_address}.
    - Include a break around lunchtime for a small snack or coffee with a pastry.
    - The tour should give a good idea of what the city has to offer without being too strenuous.
    """
    
    SUN_TEMPLATE = """
    - Create a detailed plan for the last day in {city} on Sunday.
    - Start with breakfast around 9 AM at the hotel, followed by packing luggage and checking out of the {hotel} by 12 PM (assuming the hotel can store luggage).
    - Suggest a small, relaxing walk in the city, like a promenade along a river, a park, or another convenient and scenic place or neighborhood.
    - Include a suggestion for a final drink in a typical local cafe or bar.
    - The plan should conclude with a return to the {hotel} by 3 PM to collect luggage for the trip home.
    """

    # --- ASSEMBLE THE FINAL PROMPT *INSIDE* THE FUNCTION ---
    # This is the corrected logic: the prompt is built dynamically with fresh data for each call.
    weekend_prompt = f"""
    You are a helpful travel assistant. Create a nicely formatted, day-by-day plan for a short weekend stay in {city}.
    The user is staying at the hotel "{hotel_name}" located at "{hotel_address}".
    Use the {search_content} for inspiration.
    The tone should be welcoming, detailed, and relaxed.

    Format your plan EXACTLY like this:

    ## Friday Evening: Arrival
    Design a plan for the first evening, starting after arrival at 5 PM.
    {EVENING_TEMPLATE.format(hotel=hotel_name, hotel_address=hotel_address, restaurant_list=restaurant_list )}

    ## Saturday: Exploration
    Create a relaxed, full-day plan from morning until evening, avoiding strenuous physical activities.
    {ALL_DAY_TEMPLATE.format(city=city, hotel=hotel_name, hotel_address=hotel_address)}
    After the day's activities, create a new and different evening plan starting around 6 PM.
    {EVENING_TEMPLATE.format(hotel=hotel_name, hotel_address=hotel_address, restaurant_list=restaurant_list)}

    ## Sunday: Returning home
    Create a plan for the final day.
    {SUN_TEMPLATE.format(city=city, hotel=hotel_name)}

    Ensure the plans are cohesive and do not suggest the same major activity twice (e.g., don't visit the same landmark on two different days). 
    The two evening plans should suggest different walks and restaurants.

    **CRITICAL:**
    The response must terminate immediately after the Sunday plan.
    No text may appear after the itinerary.
    No concluding paragraph.
    No closing remarks.
    No suggestions for further assistance.
    Do not include restaurants with non eurpean cuisine.
    """
    print(weekend_prompt)
    todos = get_info( weekend_prompt, use_search=False)
    print(todos)
    
    to_pdf['planner']= todos
    return m._repr_html_(), todos

#--- Create pdf report---

def remove_top_level_title(text: str) -> str:
    """
    Remove first title from generated city guide.

    Example:

    ## Discover Aachen: Your Weekend Guide

    becomes removed.
    """

    if not text:
        return ""

    lines = text.splitlines()

    if lines and lines[0].strip().startswith("#"):
        lines = lines[1:]

    return "\n".join(lines).strip()


def html_links_to_md(html_text: str) -> str:
    """Convert <a href='url'>text</a> to [text](url) for pandoc/LaTeX compatibility."""
    if not html_text:
        return ""
    
    def replace_link(match):
        url = match.group(1)
        label = match.group(2)
        # Strip any nested HTML tags from the label
        label = re.sub(r'<[^>]+>', '', label).strip()
        # Encode & in URLs so LaTeX doesn't choke
        url = url.replace('&', '%26')
        return f'[{label}]({url})'
    
    result = re.sub(
        r'<a\s+href=["\']([^"\']+)["\'][^>]*>(.*?)</a>',
        replace_link,
        html_text,
        flags=re.IGNORECASE | re.DOTALL
    )
    return result

def clean_markdown(text: str) -> str:

    if not text:
        return ""

    text = re.sub(r"\n{3,}", "\n\n", text)

    return html_links_to_md(text.strip())


def sanitize_for_pandoc(text: str) -> str:
    """Fix patterns that break pandoc's YAML/markdown parser."""
    if not text:
        return ""
    # Escape standalone section headers that start with ### followed by W (typo in your code: ###W irtshaus)
    text = re.sub(r'^(#{1,6})\s*W\s+', r'\1 W', text, flags=re.MULTILINE)
    # Remove any accidental leading --- that pandoc would treat as YAML delimiter
    text = re.sub(r'^\-\-\-\s*$', r'', text, flags=re.MULTILINE)
    return text

def build_markdown_document(data):
    city_guide = remove_top_level_title(data.get("city_guide", ""))
    sections = []
    sections.append(f"# City Guide\n\n{city_guide}")
    sections.append(f"# Tourist Office\n\n{data.get('tourist_office', '')}")
    sections.append(f"# Hotel\n\n {data.get('hotel_name','')}\n\n{data.get('hotel_description','')}")
    sections.append(f"# Restaurants\n\n{data.get('restaurants', '')}")
    sections.append(f"# Journey\n\n## Outbound Journey\n\n{data.get('trip_out','')}\n\n## Return Journey\n\n{data.get('trip_home','')}")
    sections.append(f"# Planner\n\n{data.get('planner','')}")

    document = "\n\n".join(sections)
    document = sanitize_for_pandoc(document)
    document = html_links_to_md(document)
    return clean_markdown(document)


def create_latex_template():


    template = r"""


    \documentclass[11pt,a4paper]{article}
    
    % =====================================================
    % PAGE LAYOUT
    % =====================================================
    
    \usepackage[a4paper,margin=1in]{geometry}
    
    % =====================================================
    % FONTS
    % =====================================================
    
    \usepackage{fontspec}
    \usepackage{unicode-math}
    
    \IfFontExistsTF{TeX Gyre Pagella}{
    \setmainfont{TeX Gyre Pagella}
    \setmathfont{TeX Gyre Pagella Math}
    }{
    \setmainfont{Times New Roman}
    \setmathfont{Latin Modern Math}
    }
    
    % =====================================================
    % PACKAGES
    % =====================================================
    
    \usepackage{graphicx}
    \usepackage{booktabs}
    \usepackage{longtable}
    \usepackage{array}
    \usepackage{float}
    \usepackage{calc}
    \usepackage{multicol}
    \usepackage{pdflscape}
    \usepackage{enumitem}
    \usepackage{xcolor}
    \usepackage{titlesec}
    \usepackage[hidelinks]{hyperref}
    \usepackage{fancyhdr}
    \usepackage{needspace}
    
    % Pandoc compatibility
    
    \providecommand{\tightlist}{
    \setlength{\itemsep}{0pt}
    \setlength{\parskip}{0pt}
    }
    
    \providecommand{\pandocbounded}[1]{#1}
    
    % Longtable fixes
    
    \setlength\LTleft{0pt}
    \setlength\LTright{0pt}
    
    % =====================================================
    % COLORS
    % =====================================================
    
    \definecolor{travelblue}{HTML}{003366}
    \definecolor{travelgold}{HTML}{A07F40}
    
    % =====================================================
    % SECTION NUMBERING
    % =====================================================
    
    \setcounter{secnumdepth}{4}
    \setcounter{tocdepth}{3}
    
    \titleformat{\section}
    {\Large\bfseries\color{travelblue}}
    {\thesection}
    {1em}
    {}
    
    \titleformat{\subsection}
    {\large\bfseries}
    {\thesubsection}
    {1em}
    {}
    
    \titleformat{\subsubsection}
    {\normalsize\bfseries}
    {\thesubsubsection}
    {1em}
    {}
    
    % =====================================================
    % SPACING
    % =====================================================
    
    \setlength{\parindent}{0pt}
    \setlength{\parskip}{0.75em}
    
    \setlist{nosep}
    \setlist[itemize]{leftmargin=*}
    
    % Avoid ugly page breaks
    
    \clubpenalty=10000
    \widowpenalty=10000
    \displaywidowpenalty=10000
    
    \AtBeginEnvironment{itemize}{\needspace{4\baselineskip}}
    \AtBeginEnvironment{enumerate}{\needspace{4\baselineskip}}
    
    % =====================================================
    % HEADER / FOOTER
    % =====================================================
    
    \pagestyle{fancy}
    \fancyhf{}
    
    \fancyhead[R]{\small $title$}
    
    \fancyfoot[C]{\thepage}
    
    \renewcommand{\headrulewidth}{1.2pt}
    
    \renewcommand{\headrule}{
    {\color{travelgold}
    \hrule width\headwidth height\headrulewidth
    \vskip-\headrulewidth}
    }
    
    % =====================================================
    % HYPERLINKS
    % =====================================================
    
    \hypersetup{
    colorlinks=true,
    linkcolor=travelblue,
    urlcolor=travelblue,
    citecolor=travelblue
    }
    
    % =====================================================
    % IMAGE HANDLING
    % =====================================================
    
    \floatplacement{figure}{H}
    \floatplacement{table}{H}
    
    % =====================================================
    % DOCUMENT
    % =====================================================
    
    \begin{document}
    
    % =====================================================
    % COVER PAGE
    % =====================================================
    
    \begin{titlepage}
    
    \centering

    $if(cover-image)$
    \vspace*{1cm}
    \includegraphics[width=\textwidth, height=0.35\textheight, keepaspectratio]{$cover-image$}
    \vspace{1cm}
    $else$
    \vspace*{3cm}
    $endif$
    
    {\Huge\bfseries $title$\par}
    
    \vspace{1cm}
    
    {\Large Weekend Travel Guide\par}
    
    \vspace{1.5cm}
    
    {\Large $date$\par}
    
    \vfill
    
    \Large Generated by DBG Travel
    
    \vspace{0.5cm}
    
    
    \end{titlepage}
    
    % =====================================================
    % TOC
    % =====================================================
    
    \pagenumbering{roman}
    
    \tableofcontents
    
    \newpage
    
    \pagenumbering{arabic}
    
    % =====================================================
    % BODY
    % =====================================================
    
    $body$
    
    \end{document}
    """
    

    with open(
        "travel_template.tex",
        "w",
        encoding="utf-8"
    ) as f:
        f.write(template)

def markdown_to_pdf(markdown_text, cover_image_path=None):
    try:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_pdf = f"WeekendGuide_{timestamp}.pdf"

        city = to_pdf.get('city', 'Weekend Guide')
        country = to_pdf.get('country', '')
        date_str = datetime.now().strftime("%B %d, %Y")

        yaml_header = f"""---
title: "Weekend Guide {city} {country}"
date: "{date_str}"
---

"""
        full_markdown = yaml_header + markdown_text

        with tempfile.TemporaryDirectory() as tmpdir:
            md_file = os.path.join(tmpdir, "guide.md")
            with open(md_file, "w", encoding="utf-8") as f:
                f.write(full_markdown)

            extra_args = [
                "--standalone",
                "--pdf-engine=xelatex",
                "--template=travel_template.tex",
                "--toc",
                "--toc-depth=3",
                "--number-sections",
                "--wrap=none",
            ]

            if cover_image_path and os.path.exists(cover_image_path):
                img_ext = os.path.splitext(cover_image_path)[1]
                img_dest = os.path.join(tmpdir, f"cover_image{img_ext}")
                shutil.copy(cover_image_path, img_dest)
                # Convert backslashes to forward slashes for LaTeX
                img_dest_latex = img_dest.replace("\\", "/")
                extra_args.append(f"--variable=cover-image:{img_dest_latex}")

            pypandoc.convert_file(
                md_file,
                "pdf",
                format="markdown",
                outputfile=output_pdf,
                extra_args=extra_args
            )

        return output_pdf

    except Exception as e:
        raise gr.Error(f"PDF generation failed:\n\n{str(e)}")

def generate_markdown():

    return build_markdown_document(to_pdf)
    

def update_preview(markdown_text):

    return markdown_text

create_latex_template()



def pdf_to_html(pdf_path: str) -> str:
    if not pdf_path or not os.path.exists(pdf_path):
        return "<div style='text-align:center; padding:20px;'>No PDF generated yet.</div>"

    try:
        with open(pdf_path, "rb") as f:
            base64_pdf = base64.b64encode(f.read()).decode("utf-8")

        return f"""
            <embed 
                src="data:application/pdf;base64,{base64_pdf}"
                width="100%"
                height="1000px"
                type="application/pdf"
            >
        """
    except Exception as e:
        return f"Error reading PDF: {e}"


# --- Gradio UI Definition ---
custom_css = """
    body { font-family: Verdana, Arial, sans-serif; width: 95% !important; margin: auto !important; } 
    .gradio-html iframe { height: 600px !important; }"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("<h1>Weekend Trip Planner</h1>")
    
    with gr.Tabs():
        # --- TAB 1: City & Hotel Explorer (NEW) ---
        with gr.TabItem("City & Hotel Explorer"):
            gr.Markdown("Discover a new city. Get a travel guide, find the best hotels, and then analyze your hotel's surroundings in the next tab.")
            
            # State objects to hold data between steps
            tourist_office_state = gr.State()
            found_hotels_state = gr.State()
            found_parkings_state= gr.State()
            city_name_state = gr.State()
            selected_hotel = gr.State()
            search_content = gr.State()

            with gr.Row():
                with gr.Column(scale=1):
                    city_input = gr.Textbox(label="City Name")
                    country_input = gr.Textbox(label = 'Country Name')
                    gr.Markdown('Create City Exploration Guide')
                    city_search_btn = gr.Button("Explore City", icon='icons/city-solid-full.svg')
                    hotel_line1 = gr.Markdown("---",visible=False)
                    hotel_line2 = gr.Markdown('Search for and Select Hotels',visible=False)                  
                    hotel_search_btn = gr.Button("Search Hotels", icon='icons/bed-solid-full.svg', visible=False)
                    hotel_selector_radio = gr.Radio(label="Select a Hotel to Use in Next Tab", visible=False)
                    selection_confirmation_md = gr.Markdown("") 
                    parking_line1 = gr.Markdown("---", visible=False)
                    parking_line2 = gr.Markdown('Search for Parking Lots',visible=False)   
                    parking_search_btn = gr.Button("Search Parkings", icon='icons/square-parking-solid-full.svg', visible=False)                  
                    gr.Markdown("---")                 
                             
                with gr.Column(scale=3):
                    city_map_output = gr.HTML(create_initial_map(), label="City Map")
                    city_guide_output = gr.Markdown("Your city guide will appear here...", show_copy_button=True)
            
        # --- TAB 2: EV Multi-Stop Trip Planner ---
        with gr.TabItem("EV Multi-Stop Trip Planner"):
            gr.Markdown("1. Find an initial route. 2. Select charging stops. 3. Plan the multi-stop trip.")
            found_stations_state = gr.State([])
            with gr.Row():
                with gr.Column(scale=1):
                    start_address_input = gr.Textbox(label="Start Address", placeholder="start")
                    end_address_input = gr.Textbox(label="Destination Address", placeholder="dest")
                    start_battery_input = gr.Textbox(label="Battery at start (%)", placeholder="%")
                    find_button = gr.Button("Find Route & Stations", icon = 'icons/charging-station-solid-full.svg')
                    gr.Markdown("---")
                    station_selector_out = gr.CheckboxGroup([], label="Select Stops for Way Out", visible=False)
                    station_selector_home = gr.CheckboxGroup([], label="Select Stops for Way Home", visible=False)
                    recalculate_button = gr.Button("Plan Trip with Selected Stops", icon = 'icons/car-solid-full.svg', visible=False)
                with gr.Column(scale=3):
                    gr.Markdown("### Initial Route and All Available Stations")
                    map_output = gr.HTML(create_initial_map(), label="Initial Trip Map")
                    initial_status_output = gr.Markdown("Route information will appear here.")
            
                    start_address_input = gr.Textbox(label="Start Address", value="Heirweg 85A, 9190 Stekene, Belgium", placeholder="start")
                        with gr.Column(scale=1):
                            gr.Markdown("### Way Out Details")
                            map_output_out = gr.HTML(label="Way Out Map")
                            way_out = gr.Markdown("Trip plan for the way out will be generated here.", show_copy_button=True)
                        with gr.Column(scale=1):
                            gr.Markdown("### Way Home Details")
                            map_output_home = gr.HTML(label="Way Home Map")
                            way_home = gr.Markdown("Trip plan for the way home will be generated here.", show_copy_button=True)
        
        # --- TAB 3: Hotel & Restaurant Finder ---
        with gr.TabItem("Restaurants"):
            gr.Markdown("Enter your hotel and desired cuisines to find nearby restaurants.")
            with gr.Row():
                with gr.Column(scale=1):
                    # This component will be updated by the first tab
                    hotel_address_input = gr.Textbox(label="Hotel Address", placeholder="Hotel")
                    restaurant_type_input = gr.CheckboxGroup(["Steakhouse", "Grill", "Seafood", "Belgian", "Italian", "French", "German", "Croatian", "Spanish", "Greek", "Mediterranean","Any"], label="Restaurant Type", value="Any")
                    find_restaurant_button = gr.Button("Find Restaurants",icon='icons/utensils-solid-full.svg')
                with gr.Column(scale=3):
                    restaurant_map_output = gr.HTML(create_initial_map(), label="Nearby Places Map")
                    restaurant_list_output = gr.Markdown("", show_copy_button=True)

        # --- TAB 4: Exploring the City ---            
        with gr.TabItem("Explorating the city"):
            gr.Markdown("Things to do")
            with gr.Row():
                with gr.Column(scale=1):
                    # This component will be updated by the first tab
                    todo_address_input = gr.Textbox(label="Hotel Address", placeholder="Hotel")
                    restaurant_type_input = gr.CheckboxGroup(["Local", "Italian", "Croatian", "Grill", "Steakhouse", "Seafood"], label="Restaurant Type", value=["Local", "Italian", "Croatian", "Grill", "Steakhouse", "Seafood"])
                with gr.Column(scale=3):
                    todo_map_output = gr.HTML(create_initial_map(), label="Nearby Places Map")
                    todo_output = gr.Markdown("", show_copy_button=True)

        # --- TAB 5: Generating the Report --- 
        with gr.TabItem("PDF Editor"):
            gr.Markdown("Markdown Editor & PDF Generator")
            with gr.Row():
                with gr.Column(scale=1):
                    btn_generate_md = gr.Button("Generate Markdown From Current Trip")
                    btn_generate_pdf = gr.Button("Generate Professional PDF")
                with gr.Column(scale=3):
                    cover_image = gr.Image(label="Cover Page Image (optional)", type="filepath")  # ADD THIS
            with gr.Row():
                pdf_preview = gr.HTML(label="PDF Preview")
                markdown_editor = gr.Code(language="markdown", label="Markdown Source", lines=40)
                
                #markdown_preview = gr.Markdown(label="Preview")
            pdf_output = gr.File(label="Download PDF")
            
    # --- Event Handlers ---

    # Handlers for Tab 1
    city_search_btn.click(
        fn=show_city_info, 
        inputs=[city_input,country_input], 
        outputs=[city_guide_output, search_content, city_map_output, tourist_office_state, hotel_line1, hotel_line2, hotel_search_btn, parking_line1, parking_line2,parking_search_btn, hotel_selector_radio, selection_confirmation_md]
    ).success(
        lambda x: x, city_input, city_name_state # Save city name to state on success
    )    
    hotel_search_btn.click(
        fn=search_and_display_hotels,
        inputs=[city_name_state, tourist_office_state, found_parkings_state],
        outputs=[hotel_selector_radio, found_hotels_state, city_map_output]
    )
    hotel_selector_radio.change(
        fn=store_and_update_hotel,
        inputs=[hotel_selector_radio, found_hotels_state],
        outputs=[selection_confirmation_md, hotel_address_input, end_address_input, todo_address_input, selected_hotel] 
    )
    parking_search_btn.click(
        fn=search_and_display_parkings,
        inputs=[city_name_state, tourist_office_state, found_hotels_state],
        outputs=[found_parkings_state, city_map_output]
    )

    # Handlers for Tab 2
    find_button.click(
        fn=generate_trip_map, 
        inputs=[start_address_input, end_address_input, start_battery_input], 
        outputs=[map_output, station_selector_out, station_selector_home, initial_status_output, found_stations_state, recalculate_button]
    )
    recalculate_button.click(
        fn=plan_trip_with_stops, 
        inputs=[station_selector_out, station_selector_home, found_stations_state, start_address_input, end_address_input, start_battery_input], 
        outputs=[map_output_out, way_out, map_output_home, way_home]
    )
    for comp in [start_address_input, end_address_input, start_battery_input]:
        comp.change(fn=reset_planning_ui_full, inputs=[], outputs=[station_selector_out, station_selector_home, recalculate_button, initial_status_output, way_out, way_home, map_output_out, map_output_home])


    # Handlers for Tab 3
    find_restaurant_button.click(
        fn=generate_restaurant_map_with_link,
        inputs=[hotel_address_input, restaurant_type_input,selected_hotel],
        outputs=[restaurant_map_output, restaurant_list_output])
    
    # Handlers for Tab 4
    todo_btn.click(
        fn=get_suggestions,
        inputs=[todo_address_input, selected_hotel, restaurant_list_output,search_content],
        outputs=[todo_map_output, todo_output])

    # Handlers for Tab 5
    btn_generate_md.click(
        fn=generate_markdown,
        outputs=markdown_editor
    )
    
   #markdown_editor.change(
   #     fn=update_preview,
   #     inputs=markdown_editor,
   #     outputs=markdown_preview
   # )
    
    btn_generate_pdf.click(
        fn=markdown_to_pdf,
        inputs=[markdown_editor, cover_image],  # ADD cover_image
        outputs=pdf_output
    ).success(
        fn=pdf_to_html,
        inputs=pdf_output,
        outputs=pdf_preview
    )


if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "C:\Users\dirkb\anaconda3\envs\llms\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dirkb\anaconda3\envs\llms\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\dirkb\anaconda3\envs\llms\Lib\site-packages\fastapi\applications.py", line 1054, in __call__
    await super().__call__(scope, receive, send)
  File "C:\Users\dirkb\anaconda3\envs\llms\Lib\site-packages\starlette\applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "C:\Users\dirkb\anaconda3\envs\llms\Lib\site-packages\starlette\middleware\errors.py", line 187, in __call__
    raise exc
  File "C